In [1]:
# Dependencies and SparkSession Configuration

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np
from datetime import datetime

# 16 CPUS / 128 GB Memory
spark = SparkSession.builder \
    .appName("PushshiftRedditEDA") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.executor.memory", "16g") \
    .config("spark.executor.instances", 6) \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

Matplotlib created a temporary cache directory at /scratch/ekim18/job_48518547/matplotlib-har78syr because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


Spark version: 3.5.0
Spark UI: http://exp-17-56.expanse.sdsc.edu:4040


In [4]:
# Data Load

import os
import glob
from pyspark.sql import functions as F

DATA_DIR = "/expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/"
files = sorted(glob.glob(DATA_DIR + "*.parquet"))

COLS = ["author", "created_utc", "id", "num_comments", "score", 
        "selftext", "subreddit", "subreddit_id", "title"]

def read_one(path):
    return spark.read.parquet(path) \
        .select(*[F.col(c) for c in COLS if c != "created_utc"],
                F.col("created_utc").cast("long").alias("created_utc"))

# Read and union all files
dfs = [read_one(f) for f in files]
df = dfs[0]
for d in dfs[1:]:
    df = df.unionByName(d)

print(f"Files loaded: {len(files)}")
print(f"Partitions: {df.rdd.getNumPartitions()}")

Files loaded: 218
Partitions: 3339


In [6]:
# Row Count

row_count = df.count()
print(f"Total observations: {row_count:,}")

Total observations: 549,662,955


In [7]:
# SparkUI Screenshot Cell 
# Must be run after data load and action has been triggrered
import requests

# Get the active Spark Context and URL
sc = spark.sparkContext
url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/executors"

# Fetch the executor data from the API
response = requests.get(url)
executors = response.json()

# Format into a readable DataFrame
executor_df = pd.DataFrame(executors)[['id', 'totalCores', 'maxMemory', 'activeTasks', 'isActive']]
executor_df['maxMemory_GB'] = (executor_df['maxMemory'] / (1024**3)).round(2)
executor_df

,id,totalCores,maxMemory,activeTasks,isActive,maxMemory_GB
0,driver,16,4965217075,0,True,4.62


In [8]:
print(spark.sparkContext.master)

local[*]
